# EDA — Liquidation Signal (Binance trades / BBO / Binance & Bybit liquidations)

Цель ноутбука — разведать данные (3 месяца BTC и ETH perpetual) перед построением сигнала-фильтра
для maker-стратегии. Структура повторяет вопросы из `description.md`:

1. **Shape of the data** — объёмы, покрытие, плотность, gaps/duplicates.
2. **Distributions** — цены, объёмы, нотионалы, спрэды, баланс сторон, тяжёлые хвосты.
3. **Cross-source relationships** — как ведёт себя книга вокруг сделок и ликвидаций; синхронизация Binance/Bybit.
4. **Conventions that bite** — единицы времени, `side` в трейдах/ликвидациях, задержка Bybit +200 ms.
5. **Anything weird** — выбросы, неожиданные паузы, странные значения.
6. **Markout / baseline** — мини-пайплайн расчёта `pnl_i(τ)` и таблица `PnL_all` по дням.

Данные большие (BTC trades ≈ 400 M строк, BBO ≈ 100 M), поэтому везде, где возможно, используется
**стриминговое чтение** через `pyarrow.dataset` с predicate pushdown и сэмплирование row group'ами.


## 0. Setup, imports, helpers

In [ ]:
import os, sys, math, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi":     110,
    "axes.grid":      True,
    "grid.alpha":     0.3,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

DATA = Path("data")  # relative to notebook
assert DATA.exists(), f"data/ not found at {DATA.resolve()}"

PATHS = {
    "binance_trades_btc": DATA / "binance_trades"       / "perp_btcusdt.parquet",
    "binance_trades_eth": DATA / "binance_trades"       / "perp_ethusdt.parquet",
    "binance_bbo_btc":    DATA / "binance_booktickers"  / "perp_btcusdt.parquet",
    "binance_bbo_eth":    DATA / "binance_booktickers"  / "perp_ethusdt.parquet",
    "binance_liq_btc":    DATA / "binance_liquidations" / "perp_btcusdt.parquet",
    "binance_liq_eth":    DATA / "binance_liquidations" / "perp_ethusdt.parquet",
    "bybit_liq_btc":      DATA / "bybit_liquidations"   / "btcusdt.parquet",
    "bybit_liq_eth":      DATA / "bybit_liquidations"   / "ethusdt.parquet",
}
for k, p in PATHS.items():
    assert p.exists(), f"missing {k}: {p}"
print("All parquet files present.")


In [ ]:
# ----- helpers -----------------------------------------------------------

US_PER_S   = 1_000_000
US_PER_MS  =      1_000
US_PER_DAY = 24 * 3600 * US_PER_S

def us_to_dt(ts_us, tz="UTC"):
    # int64 microseconds -> pandas Timestamp/Series in UTC.
    out = pd.to_datetime(ts_us, unit="us", utc=True)
    if tz == "UTC":
        return out
    # Series -> .dt.tz_convert ; DatetimeIndex/scalar -> .tz_convert
    return out.dt.tz_convert(tz) if hasattr(out, "dt") else out.tz_convert(tz)

def day_range(path):
    # Return (t_min, t_max) in microseconds from parquet stats. Instant, no read.
    pf = pq.ParquetFile(path)
    mins, maxs = [], []
    for i in range(pf.num_row_groups):
        rg = pf.metadata.row_group(i)
        for j in range(rg.num_columns):
            col = rg.column(j)
            if col.path_in_schema == "timestamp":
                s = col.statistics
                if s is not None and s.has_min_max:
                    mins.append(s.min); maxs.append(s.max)
    return (min(mins), max(maxs)) if mins else (None, None)

def read_window(path, t_us_start, t_us_end, columns=None):
    # Read rows with t_us_start <= timestamp < t_us_end via predicate pushdown.
    dset = ds.dataset(str(path))
    flt  = (ds.field("timestamp") >= t_us_start) & (ds.field("timestamp") < t_us_end)
    tbl  = dset.to_table(filter=flt, columns=columns)
    return tbl.to_pandas()

def read_day(path, day, columns=None):
    # day: e.g. "2025-12-15" (interpreted as UTC).
    t0 = int(pd.Timestamp(day, tz="UTC").value // 1000)  # ns -> us
    return read_window(path, t0, t0 + US_PER_DAY, columns=columns)

def read_sample_rowgroups(path, max_rows=400_000, columns=None, seed=0):
    # Sample roughly `max_rows` rows by picking random row groups.
    pf = pq.ParquetFile(path)
    n_rg = pf.num_row_groups
    if n_rg == 0:
        return pd.DataFrame()
    sizes = np.array([pf.metadata.row_group(i).num_rows for i in range(n_rg)])
    avg   = max(int(sizes.mean()), 1)
    k     = max(1, min(n_rg, math.ceil(max_rows / avg)))
    rng   = np.random.default_rng(seed)
    pick  = sorted(rng.choice(n_rg, size=k, replace=False).tolist())
    tbl   = pf.read_row_groups(pick, columns=columns)
    df    = tbl.to_pandas()
    if len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=seed)
    return df.sort_values("timestamp").reset_index(drop=True)

def stream_groupby_day(path, agg="count", columns=("timestamp",), batch_size=2_000_000):
    # Stream the file and return daily counts (or per-day sum of notional).
    cols = list(columns)
    counts = {}
    for batch in ds.dataset(str(path)).to_batches(batch_size=batch_size, columns=cols):
        ts = batch.column("timestamp").to_numpy()
        day_idx = ts // US_PER_DAY  # integer day since epoch
        if agg == "count":
            uniq, cnt = np.unique(day_idx, return_counts=True)
            for d, c in zip(uniq.tolist(), cnt.tolist()):
                counts[d] = counts.get(d, 0) + c
        elif agg == "amount_notional":
            price  = batch.column("price").to_numpy()
            amount = batch.column("amount").to_numpy()
            notional = price * amount
            order = np.argsort(day_idx, kind="stable")
            di = day_idx[order]; nt = notional[order]
            uniq, idx = np.unique(di, return_index=True)
            sums = np.add.reduceat(nt, idx)
            for d, s in zip(uniq.tolist(), sums.tolist()):
                counts[d] = counts.get(d, 0.0) + s
    df = pd.DataFrame({"day_idx": list(counts), "value": list(counts.values())})
    df["date"]  = pd.to_datetime(df["day_idx"] * US_PER_DAY, unit="us", utc=True)
    return df.sort_values("date").reset_index(drop=True)

def humanize(n):
    for u in ("", "K", "M", "B"):
        if abs(n) < 1000: return f"{n:.2f}{u}"
        n /= 1000
    return f"{n:.2f}T"


In [ ]:
# Quick sanity printout — counts, time ranges, file size.
META = {}
for k, p in PATHS.items():
    pf = pq.ParquetFile(p)
    tmin, tmax = day_range(p)
    META[k] = dict(
        rows         = pf.metadata.num_rows,
        row_groups   = pf.num_row_groups,
        size_bytes   = p.stat().st_size,
        t_min        = tmin, t_max = tmax,
        t_min_human  = str(us_to_dt(tmin)) if tmin is not None else None,
        t_max_human  = str(us_to_dt(tmax)) if tmax is not None else None,
    )
meta_df = pd.DataFrame(META).T
meta_df["rows_h"]  = meta_df["rows"].apply(humanize)
meta_df["size_mb"] = (meta_df["size_bytes"] / 1e6).round(1)
meta_df[["rows_h", "row_groups", "size_mb", "t_min_human", "t_max_human"]]


## 1. Shape of the data

Сколько у нас событий на источник × символ × день? Равномерно ли они распределены? Есть ли gaps/duplicates?


In [ ]:
# Per-day event count for every source (streamed — обходим файл по батчам).
# Это занимает ~30-90 секунд на каждом из больших файлов.
daily = {}
for k, p in PATHS.items():
    daily[k] = stream_groupby_day(p, agg="count").rename(columns={"value": "n_events"})
    print(f"{k:25s}  days={len(daily[k]):3d}  total={int(daily[k].n_events.sum()):,}")


In [ ]:
# Plot — events per day (одна панель на каждый источник).
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
ax_iter = axes.flatten()
for ax, (k, df) in zip(ax_iter, daily.items()):
    ax.plot(df["date"], df["n_events"], lw=1.2)
    ax.set_title(k, fontsize=10)
    ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Events per UTC day, log scale", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Per-day TOTAL NOTIONAL для trades/liquidations.
notional = {}
for k in ["binance_trades_btc", "binance_trades_eth",
          "binance_liq_btc",    "binance_liq_eth",
          "bybit_liq_btc",      "bybit_liq_eth"]:
    notional[k] = stream_groupby_day(
        PATHS[k], agg="amount_notional",
        columns=("timestamp", "price", "amount"))
    notional[k]["notional_usd"] = notional[k]["value"]
    print(f"{k:25s}  avg daily notional = ${notional[k].notional_usd.mean()/1e6:,.1f} M")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
groups = [
    ("binance_trades_btc", "binance_trades_eth"),
    ("binance_liq_btc",    "binance_liq_eth"),
    ("bybit_liq_btc",      "bybit_liq_eth"),
]
for row, (a, b) in enumerate(groups):
    for col, k in enumerate((a, b)):
        ax = axes[row, col]
        ax.plot(notional[k]["date"], notional[k]["notional_usd"] / 1e6, lw=1.2)
        ax.set_title(k + "   (USD M / day)")
        ax.tick_params(axis="x", rotation=30)
fig.suptitle("Notional turnover per UTC day", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Intraday pattern — events per hour-of-day на одной неделе в середине train-периода.
WEEK_START = "2026-01-12"
WEEK_END   = "2026-01-19"
t0 = int(pd.Timestamp(WEEK_START, tz="UTC").value // 1000)
t1 = int(pd.Timestamp(WEEK_END,   tz="UTC").value // 1000)

def hour_hist(path, t0, t1):
    counts = np.zeros(24, dtype=np.int64)
    for batch in ds.dataset(str(path)).to_batches(
            batch_size=2_000_000, columns=["timestamp"],
            filter=(ds.field("timestamp") >= t0) & (ds.field("timestamp") < t1)):
        ts = batch.column("timestamp").to_numpy()
        h  = (ts // (3600 * US_PER_S)) % 24
        np.add.at(counts, h, 1)
    return counts

hours = {k: hour_hist(p, t0, t1) for k, p in PATHS.items()}

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)
panels = {
    "Binance trades":  ["binance_trades_btc", "binance_trades_eth"],
    "Binance BBO":     ["binance_bbo_btc",    "binance_bbo_eth"],
    "Binance liq":     ["binance_liq_btc",    "binance_liq_eth"],
    "Bybit liq":       ["bybit_liq_btc",      "bybit_liq_eth"],
}
for ax, (title, keys) in zip(axes.flatten(), panels.items()):
    for off, k in enumerate(keys):
        ax.bar(np.arange(24) + off*0.35, hours[k], width=0.35, label=k)
    ax.set_title(title); ax.set_xlabel("hour of day, UTC"); ax.legend(fontsize=8)
fig.suptitle(f"Hourly event count, week {WEEK_START} → {WEEK_END}", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Inter-arrival time распределение — оцениваем burstiness.
DAY = "2026-01-15"

def iat_microseconds(path, day):
    df = read_day(path, day, columns=["timestamp"])
    if df.empty: return np.array([])
    ts = np.sort(df["timestamp"].values)
    return np.diff(ts)

iats = {}
for k in ["binance_trades_btc", "binance_liq_btc", "bybit_liq_btc",
          "binance_trades_eth", "binance_liq_eth", "bybit_liq_eth",
          "binance_bbo_btc",    "binance_bbo_eth"]:
    iats[k] = iat_microseconds(PATHS[k], DAY)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
groupings = [
    ["binance_trades_btc", "binance_trades_eth"],
    ["binance_bbo_btc",    "binance_bbo_eth"],
    ["binance_liq_btc",    "binance_liq_eth"],
    ["bybit_liq_btc",      "bybit_liq_eth"],
]
for ax, keys in zip(axes.flatten(), groupings):
    for k in keys:
        x = iats[k]
        if len(x) == 0: continue
        ax.hist(np.clip(x, 1, None), bins=np.logspace(0, 9, 80), label=k, alpha=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("inter-arrival time, µs (log)")
    ax.legend(fontsize=8)
fig.suptitle(f"Inter-arrival times on {DAY}", y=1.02)
plt.tight_layout()
plt.show()

iat_stats = pd.DataFrame({k: pd.Series({
    "n":      len(v),
    "p50_ms": np.median(v) / US_PER_MS if len(v) else np.nan,
    "p99_ms": np.percentile(v, 99) / US_PER_MS if len(v) else np.nan,
    "max_s":  v.max() / US_PER_S if len(v) else np.nan,
}) for k, v in iats.items()}).T
iat_stats


In [ ]:
# Duplicates / monotonicity.
def dup_audit(path, day):
    df = read_day(path, day)
    n = len(df)
    same_ts   = (df["timestamp"].duplicated()).sum()
    full_dup  = (df.duplicated()).sum()
    not_mono  = (df["timestamp"].diff() < 0).sum()
    return dict(rows=n, same_ts=int(same_ts), full_dup=int(full_dup), non_monotonic=int(not_mono))

dup = {k: dup_audit(PATHS[k], DAY) for k in PATHS}
pd.DataFrame(dup).T


## 2. Distributions

Цены, объёмы, нотионалы, спрэды, баланс сторон. Сравнение BTC ↔ ETH и Binance ↔ Bybit. Хвосты.


In [ ]:
SAMPLE_DAY = "2026-01-15"

# Сэмплы для распределений — один UTC-день каждой таблицы.
sa = {k: read_day(PATHS[k], SAMPLE_DAY) for k in PATHS}
for k, df in sa.items():
    print(f"{k:25s}  rows={len(df):>10,}")


In [ ]:
# Trades — price / amount / notional / side balance.
def panel_trades(df, title, ax_price, ax_amt, ax_not, ax_side):
    df = df.copy()
    df["notional"] = df["price"] * df["amount"]
    ax_price.hist(df["price"], bins=80); ax_price.set_title(f"{title}: price")
    ax_amt.hist(np.log10(df["amount"].clip(lower=1e-9)), bins=80)
    ax_amt.set_title(f"{title}: log10(amount)")
    ax_not.hist(np.log10(df["notional"].clip(lower=1)), bins=80)
    ax_not.set_title(f"{title}: log10(notional, USD)")
    side_cnt = df["side"].value_counts()
    side_not = df.groupby("side")["notional"].sum()
    x = np.arange(2); w = 0.35
    ax_side.bar(x - w/2, side_cnt.reindex(["buy", "sell"]).values, w, label="count")
    ax_side2 = ax_side.twinx()
    ax_side2.bar(x + w/2, side_not.reindex(["buy", "sell"]).values / 1e6, w,
                 color="orange", label="notional, M$")
    ax_side.set_xticks(x); ax_side.set_xticklabels(["buy(taker)", "sell(taker)"])
    ax_side.set_title(f"{title}: side balance")

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
panel_trades(sa["binance_trades_btc"], "Binance trades BTC", *axes[0])
panel_trades(sa["binance_trades_eth"], "Binance trades ETH", *axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# Liquidations — отдельно Binance и Bybit, BTC vs ETH.
def panel_liq(df, title, ax_price, ax_not, ax_side):
    df = df.copy()
    df["notional"] = df["price"] * df["amount"]
    ax_price.hist(df["price"], bins=80); ax_price.set_title(f"{title}: price")
    ax_not.hist(np.log10(df["notional"].clip(lower=1)), bins=80)
    ax_not.set_title(f"{title}: log10(notional, USD)")
    side_cnt = df["side"].value_counts()
    side_not = df.groupby("side")["notional"].sum()
    x = np.arange(2); w = 0.35
    ax_side.bar(x - w/2, side_cnt.reindex(["buy", "sell"]).values, w, label="count")
    ax_side2 = ax_side.twinx()
    ax_side2.bar(x + w/2, side_not.reindex(["buy", "sell"]).values / 1e6, w,
                 color="orange", label="notional, M$")
    ax_side.set_xticks(x); ax_side.set_xticklabels(["buy(close-short)", "sell(close-long)"])
    ax_side.set_title(f"{title}: side balance")

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
panel_liq(sa["binance_liq_btc"], "Binance liq BTC", *axes[0])
panel_liq(sa["bybit_liq_btc"],   "Bybit liq BTC",   *axes[1])
panel_liq(sa["binance_liq_eth"], "Binance liq ETH", *axes[2])
panel_liq(sa["bybit_liq_eth"],   "Bybit liq ETH",   *axes[3])
plt.tight_layout(); plt.show()


In [ ]:
# BBO: spread в bps, размеры на топе.
def panel_bbo(df, title, ax_sp, ax_sp_log, ax_sz):
    df = df.copy()
    mid = 0.5 * (df["bid_price"] + df["ask_price"])
    spread_bps = (df["ask_price"] - df["bid_price"]) / mid * 1e4
    ax_sp.hist(spread_bps.clip(0, spread_bps.quantile(0.999)), bins=80)
    ax_sp.set_title(f"{title}: spread, bps  (clipped at p99.9)")
    ax_sp_log.hist(np.log10(spread_bps.clip(lower=1e-3)), bins=80)
    ax_sp_log.set_title(f"{title}: log10(spread bps)")
    ax_sz.hist(np.log10(df["bid_amount"].clip(lower=1e-6)), bins=80, alpha=0.5, label="bid_amt")
    ax_sz.hist(np.log10(df["ask_amount"].clip(lower=1e-6)), bins=80, alpha=0.5, label="ask_amt")
    ax_sz.set_title(f"{title}: log10(top of book size)")
    ax_sz.legend()
    return spread_bps

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
sp_btc = panel_bbo(sa["binance_bbo_btc"], "Binance BBO BTC", *axes[0])
sp_eth = panel_bbo(sa["binance_bbo_eth"], "Binance BBO ETH", *axes[1])
plt.tight_layout(); plt.show()

print("BTC spread bps   median = %.3f, p95 = %.3f, p99 = %.3f" %
      (sp_btc.median(), sp_btc.quantile(.95), sp_btc.quantile(.99)))
print("ETH spread bps   median = %.3f, p95 = %.3f, p99 = %.3f" %
      (sp_eth.median(), sp_eth.quantile(.95), sp_eth.quantile(.99)))


In [ ]:
# Heavy tails — CCDF в log-log для notional.
def ccdf_loglog(ax, vals, label):
    v = np.sort(vals[vals > 0])
    if len(v) == 0: return
    y = 1.0 - np.arange(len(v)) / len(v)
    ax.plot(v, y, lw=1, label=label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sym in zip(axes, ["btc", "eth"]):
    for src in ["binance_trades", "binance_liq", "bybit_liq"]:
        key = f"{src}_{sym}"
        df  = sa[key].copy()
        ccdf_loglog(ax, df["price"].values * df["amount"].values, key)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"{sym.upper()}  CCDF of notional, USD")
    ax.set_xlabel("notional, USD"); ax.set_ylabel("P(X > x)")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
def stats(df):
    nt = df["price"] * df["amount"]
    return pd.Series({
        "n":            len(df),
        "median$":      nt.median(),
        "p95$":         nt.quantile(0.95),
        "p99$":         nt.quantile(0.99),
        "p99.9$":       nt.quantile(0.999),
        "max$":         nt.max(),
        "sum$_M":       nt.sum() / 1e6,
    })

tbl_stats = pd.DataFrame({k: stats(sa[k]) for k in
    ["binance_trades_btc","binance_trades_eth",
     "binance_liq_btc",   "binance_liq_eth",
     "bybit_liq_btc",     "bybit_liq_eth"]}).T
tbl_stats.round(2)


In [ ]:
# Сравнение Binance vs Bybit liquidations: гистограммы log10(notional).
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, sym in zip(axes, ["btc", "eth"]):
    for src in ["binance_liq", "bybit_liq"]:
        d = sa[f"{src}_{sym}"]
        ax.hist(np.log10((d["price"] * d["amount"]).clip(lower=1)),
                bins=60, alpha=0.5, label=src)
    ax.set_title(f"{sym.upper()}: log10(liquidation notional, USD)")
    ax.legend()
plt.tight_layout(); plt.show()


## 3. Relationships across sources

Trades, BBO, liquidations лежат на одной временной оси.
Что происходит с книгой вокруг сделки? Вокруг ликвидации? Совпадают ли ликвидации Binance и Bybit?


In [ ]:
EV_DAY = "2026-01-15"
SYM = "btc"  # переключайте на "eth" если нужно

trades = sa[f"binance_trades_{SYM}"].sort_values("timestamp").reset_index(drop=True)
bbo    = sa[f"binance_bbo_{SYM}"   ].sort_values("timestamp").reset_index(drop=True)
liq_b  = sa[f"binance_liq_{SYM}"   ].sort_values("timestamp").reset_index(drop=True)
liq_y  = sa[f"bybit_liq_{SYM}"     ].sort_values("timestamp").reset_index(drop=True)

# Bybit liquidations — сдвигаем timestamp вперёд на 200 ms (как требует description.md).
liq_y_shifted = liq_y.copy()
liq_y_shifted["timestamp"] = liq_y_shifted["timestamp"] + 200 * US_PER_MS

bbo["mid"] = 0.5 * (bbo["bid_price"] + bbo["ask_price"])
bbo["spread_bps"] = (bbo["ask_price"] - bbo["bid_price"]) / bbo["mid"] * 1e4
bbo["imb"] = (bbo["bid_amount"] - bbo["ask_amount"]) / (bbo["bid_amount"] + bbo["ask_amount"])
print(len(trades), len(bbo), len(liq_b), len(liq_y_shifted))


In [ ]:
# Event study — поведение mid и imbalance в окрестности Binance-ликвидации.
def event_study(events_ts, bbo, window_us, n_grid=41, max_events=2000):
    ts_bbo  = bbo["timestamp"].values
    mid     = bbo["mid"].values
    spread  = bbo["spread_bps"].values
    imb     = bbo["imb"].values
    grid_us = np.linspace(-window_us, window_us, n_grid).astype(np.int64)

    ev = events_ts[:max_events]
    n  = len(ev); m = len(grid_us)
    out_ret = np.full((n, m), np.nan); out_sp = out_ret.copy(); out_imb = out_ret.copy()
    base_idx = np.searchsorted(ts_bbo, ev, side="right") - 1
    for i, ti in enumerate(ev):
        bi0 = base_idx[i]
        if bi0 < 0: continue
        m0 = mid[bi0]
        targets = ti + grid_us
        idx = np.searchsorted(ts_bbo, targets, side="right") - 1
        ok = idx >= 0
        if not ok.any(): continue
        out_ret[i, ok] = (mid[idx[ok]] / m0 - 1) * 1e4
        out_sp [i, ok] = spread[idx[ok]]
        out_imb[i, ok] = imb[idx[ok]]
    return grid_us, out_ret, out_sp, out_imb

liq_buy_ts  = liq_b.loc[liq_b["side"] == "buy",  "timestamp"].values
liq_sell_ts = liq_b.loc[liq_b["side"] == "sell", "timestamp"].values
WIN = 30 * US_PER_S

grid, ret_buy,  sp_buy,  imb_buy  = event_study(liq_buy_ts,  bbo, WIN)
_,    ret_sell, sp_sell, imb_sell = event_study(liq_sell_ts, bbo, WIN)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
g_s = grid / US_PER_S
axes[0].plot(g_s, np.nanmean(ret_buy,  axis=0), label="buy liq",  color="C0")
axes[0].plot(g_s, np.nanmean(ret_sell, axis=0), label="sell liq", color="C3")
axes[0].axvline(0, color="k", lw=0.5); axes[0].set_title("Mid return, bps")
axes[0].set_xlabel("seconds from event"); axes[0].legend()
axes[1].plot(g_s, np.nanmean(sp_buy,  axis=0), label="buy")
axes[1].plot(g_s, np.nanmean(sp_sell, axis=0), label="sell")
axes[1].axvline(0, color="k", lw=0.5); axes[1].set_title("Spread, bps"); axes[1].legend()
axes[2].plot(g_s, np.nanmean(imb_buy,  axis=0), label="buy")
axes[2].plot(g_s, np.nanmean(imb_sell, axis=0), label="sell")
axes[2].axvline(0, color="k", lw=0.5); axes[2].set_title("Book imbalance"); axes[2].legend()
plt.suptitle(f"Binance liquidations on {EV_DAY} — event study ({SYM.upper()})", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Event study вокруг крупных трейдов.
trades["notional"] = trades["price"] * trades["amount"]
big = trades[trades["notional"] >= trades["notional"].quantile(0.95)]
buy_big  = big.loc[big["side"] == "buy",  "timestamp"].values
sell_big = big.loc[big["side"] == "sell", "timestamp"].values

WIN = 5 * US_PER_S
grid, r_b, _, i_b = event_study(buy_big,  bbo, WIN, n_grid=41, max_events=1000)
_,    r_s, _, i_s = event_study(sell_big, bbo, WIN, n_grid=41, max_events=1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
g_s = grid / US_PER_S
axes[0].plot(g_s, np.nanmean(r_b, axis=0), label="taker buy",  color="C0")
axes[0].plot(g_s, np.nanmean(r_s, axis=0), label="taker sell", color="C3")
axes[0].axvline(0, color="k", lw=0.5)
axes[0].set_title("Mid return, bps  (large trades)"); axes[0].set_xlabel("s"); axes[0].legend()
axes[1].plot(g_s, np.nanmean(i_b, axis=0), label="taker buy")
axes[1].plot(g_s, np.nanmean(i_s, axis=0), label="taker sell")
axes[1].axvline(0, color="k", lw=0.5)
axes[1].set_title("Book imbalance"); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Bybit liquidations vs Binance — что мы видим на Binance после Bybit-ликвидации?
y_buy_ts  = liq_y_shifted.loc[liq_y_shifted["side"] == "buy",  "timestamp"].values
y_sell_ts = liq_y_shifted.loc[liq_y_shifted["side"] == "sell", "timestamp"].values

WIN = 10 * US_PER_S
grid, r_b, _, _ = event_study(y_buy_ts,  bbo, WIN, n_grid=51, max_events=2000)
_,    r_s, _, _ = event_study(y_sell_ts, bbo, WIN, n_grid=51, max_events=2000)

plt.figure(figsize=(10, 4))
plt.plot(grid / US_PER_S, np.nanmean(r_b, axis=0), label="Bybit buy liq (shifted +200ms)", color="C0")
plt.plot(grid / US_PER_S, np.nanmean(r_s, axis=0), label="Bybit sell liq (shifted +200ms)", color="C3")
plt.axvline(0, color="k", lw=0.5)
plt.title(f"Binance mid return around Bybit liquidations, {SYM.upper()} {EV_DAY}")
plt.xlabel("seconds from shifted Bybit event"); plt.ylabel("bps"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Coincidence: сколько Bybit-ликвидаций совпадают с Binance-ликвидациями того же знака в окне ±w?
def coincidence(events_a, events_b, window_us):
    b = np.sort(events_b)
    hits = 0
    for t in events_a:
        i = np.searchsorted(b, t)
        for j in (i - 1, i):
            if 0 <= j < len(b) and abs(b[j] - t) <= window_us:
                hits += 1; break
    return hits

results = {}
for sgn in ("buy", "sell"):
    a = liq_b.loc[liq_b["side"] == sgn, "timestamp"].values
    bvals = liq_y_shifted.loc[liq_y_shifted["side"] == sgn, "timestamp"].values
    for w_s in (0.2, 1.0, 5.0):
        h = coincidence(a, bvals, int(w_s * US_PER_S))
        results[(sgn, w_s)] = (h, len(a), h / max(1, len(a)))

print("Side / window(s) -> coincidences / total Binance-liq / share")
for (sgn, w), (h, n, sh) in results.items():
    print(f"  {sgn:5s}  +/-{w:4.1f}s   {h:>5d} / {n:>5d}   {sh:6.1%}")


## 4. Conventions that bite

Проверяем руками: единицы времени, `side` в трейдах и ликвидациях, задержка Bybit +200 ms.


In [ ]:
# 4.1. Timestamp unit verification.
print("First-row timestamps:")
for k, p in PATHS.items():
    pf = pq.ParquetFile(p)
    t0 = pf.read_row_group(0, columns=["timestamp"])["timestamp"][0].as_py()
    print(f"  {k:25s}  raw int = {t0}   as us UTC = {us_to_dt(t0)}")


In [ ]:
# 4.2. Trade side = taker side?
# Если так, цена сделки с side="buy" должна быть на ask, а side="sell" — на bid (BBO before trade).
def near_bbo(trades, bbo, n=20_000):
    tt = trades.sample(n=min(n, len(trades)), random_state=0).sort_values("timestamp")
    bb = bbo.sort_values("timestamp")
    merged = pd.merge_asof(
        tt, bb[["timestamp", "bid_price", "ask_price"]],
        on="timestamp", direction="backward")
    return merged.dropna(subset=["bid_price", "ask_price"])

m = near_bbo(trades, bbo)
m["on_ask"]    = (m["price"] >= m["ask_price"] - 1e-8).astype(int)
m["on_bid"]    = (m["price"] <= m["bid_price"] + 1e-8).astype(int)
m["above_mid"] = (m["price"] > 0.5*(m["bid_price"] + m["ask_price"])).astype(int)

print("trade side='buy'  -> price on ask      :", round(m.loc[m["side"]=="buy", "on_ask"].mean(), 3))
print("trade side='buy'  -> price above mid   :", round(m.loc[m["side"]=="buy", "above_mid"].mean(), 3))
print("trade side='sell' -> price on bid      :", round(m.loc[m["side"]=="sell","on_bid"].mean(), 3))
print("trade side='sell' -> price below mid   :", round((1 - m.loc[m["side"]=="sell","above_mid"]).mean(), 3))
print("If 'on ask/on bid' ratios ~1, taker-side convention is confirmed.")


In [ ]:
# 4.3. Liquidation side semantics.
def fwd_return(events_ts, bbo, horizons_us, max_events=5000):
    ts_bbo = bbo["timestamp"].values
    mid    = bbo["mid"].values
    ev = events_ts[:max_events]
    out = []
    base = np.searchsorted(ts_bbo, ev, side="right") - 1
    for h in horizons_us:
        idx = np.searchsorted(ts_bbo, ev + h, side="right") - 1
        ok = (base >= 0) & (idx >= 0)
        ret = (mid[idx[ok]] / mid[base[ok]] - 1) * 1e4
        out.append(ret)
    return out

H = [int(s * US_PER_S) for s in (1, 5, 30, 120, 300)]
labels = ["1s", "5s", "30s", "2m", "5m"]

def avg_table(name, events_ts):
    rets = fwd_return(events_ts, bbo, H)
    return pd.Series({lab: np.nanmean(r) for lab, r in zip(labels, rets)}, name=name)

rows = [
    avg_table("Binance liq buy",  liq_b.loc[liq_b["side"]=="buy",  "timestamp"].values),
    avg_table("Binance liq sell", liq_b.loc[liq_b["side"]=="sell", "timestamp"].values),
    avg_table("Bybit liq buy +200ms",  liq_y_shifted.loc[liq_y_shifted["side"]=="buy",  "timestamp"].values),
    avg_table("Bybit liq sell +200ms", liq_y_shifted.loc[liq_y_shifted["side"]=="sell", "timestamp"].values),
]
tbl = pd.DataFrame(rows).round(3)
tbl


In [ ]:
# 4.4. +200 ms Bybit delay — пик корреляции по сдвигу.
def lag_corr(liq_y_raw_ts, liq_y_raw_signs, trades, lags_us):
    tt = trades.copy()
    tt["sign"] = tt["side"].map({"buy": 1, "sell": -1})
    tt["notional"] = tt["price"] * tt["amount"]
    tt = tt.sort_values("timestamp").reset_index(drop=True)
    ts_t = tt["timestamp"].values
    sg_t = tt["sign"].values
    nt_t = tt["notional"].values

    res = []
    for lag in lags_us:
        ev = liq_y_raw_ts + lag
        i1 = np.searchsorted(ts_t, ev, side="right")
        i2 = np.searchsorted(ts_t, ev + US_PER_S, side="right")
        flows = np.array([np.dot(sg_t[a:b], nt_t[a:b]) for a, b in zip(i1, i2)])
        if flows.std() > 0 and liq_y_raw_signs.std() > 0:
            c = np.corrcoef(liq_y_raw_signs, flows)[0, 1]
        else:
            c = np.nan
        res.append(c)
    return np.array(res)

y_raw_ts    = liq_y["timestamp"].values
y_raw_sign  = liq_y["side"].map({"buy": 1, "sell": -1}).values
lags = np.arange(-1000, 2001, 50) * US_PER_MS

corrs = lag_corr(y_raw_ts, y_raw_sign, trades, lags)

plt.figure(figsize=(11, 4))
plt.plot(lags / US_PER_MS, corrs, marker="o", ms=3)
plt.axvline(200, color="r", ls="--", label="+200 ms (declared)")
plt.axhline(0,  color="k", lw=0.5)
plt.xlabel("Bybit-ts shift, ms")
plt.ylabel("corr(signed Bybit liq, signed Binance flow 1s after)")
plt.title("Cross-exchange lag scan")
plt.legend(); plt.tight_layout(); plt.show()

best = int(lags[np.nanargmax(corrs)] / US_PER_MS)
print(f"Empirical best lag ~ {best} ms   (declared 200 ms)")


## 5. Anything weird

Выбросы, странные значения, неожиданные паузы — всё, что замечаем.


In [ ]:
# 5.1. Crossed / zero spreads.
bbo_d = sa[f"binance_bbo_{SYM}"].copy()
bbo_d["spread"] = bbo_d["ask_price"] - bbo_d["bid_price"]
n0 = (bbo_d["spread"] == 0).sum()
nneg = (bbo_d["spread"] < 0).sum()
print(f"BBO rows on {EV_DAY}: {len(bbo_d):,}")
print(f"  spread == 0  : {n0:>8,}   ({n0/len(bbo_d):.2%})")
print(f"  spread <  0  : {nneg:>8,}   crossed book")
print(f"  spread > 10bp: {(bbo_d['spread']/bbo_d['bid_price']*1e4 > 10).sum():>8,}")


In [ ]:
# 5.2. Top-10 биггест трейды и ликвидации.
print("=== Top 10 trades by notional ===")
print(trades.assign(notional=lambda d: d["price"]*d["amount"])
            .sort_values("notional", ascending=False).head(10)
            [["timestamp","side","price","amount","notional"]].assign(
                ts=lambda d: us_to_dt(d["timestamp"])).to_string(index=False))

print("\n=== Top 10 liquidations by notional ===")
allq = pd.concat([liq_b.assign(src="binance"), liq_y.assign(src="bybit")], ignore_index=True)
allq["notional"] = allq["price"] * allq["amount"]
print(allq.sort_values("notional", ascending=False).head(10).assign(
    ts=lambda d: us_to_dt(d["timestamp"]))[
    ["src","timestamp","side","price","amount","notional"]].to_string(index=False))


In [ ]:
# 5.3. Долгие паузы без BBO-обновлений.
ts = bbo_d["timestamp"].values
gaps_ms = np.diff(np.sort(ts)) / US_PER_MS
print(f"BBO gap distribution (ms): p50={np.median(gaps_ms):.1f}, "
      f"p95={np.percentile(gaps_ms,95):.1f}, p99={np.percentile(gaps_ms,99):.1f}, "
      f"max={gaps_ms.max():.0f}")
plt.figure(figsize=(10, 3.5))
plt.hist(np.clip(gaps_ms, 1e-1, None), bins=np.logspace(-1, 4, 80))
plt.xscale("log"); plt.yscale("log")
plt.xlabel("BBO inter-update gap, ms"); plt.ylabel("count"); plt.title(f"BBO gaps {EV_DAY}")
plt.tight_layout(); plt.show()


In [ ]:
# 5.4. Несколько событий на одинаковый timestamp.
ts_counts = trades.groupby("timestamp").size()
print(f"Unique trade timestamps : {ts_counts.size:,}")
print(f"With >1 trades same ts  : {(ts_counts > 1).sum():,}  ({(ts_counts > 1).mean():.2%})")
print(f"Max trades on same ts   : {ts_counts.max()}")
print("Это нормально — биржа агрегирует несколько fills на одного takerа.")


## 6. Markout / baseline metric

Считаем `pnl_i(τ)` по формуле из ТЗ. Это санити-чек логики и baseline, который должен побить сигнал.


In [ ]:
def compute_markouts(trades, bbo, horizons_s=(30, 120, 300), notional_cap=100_000):
    tt = trades.sort_values("timestamp").reset_index(drop=True).copy()
    bb = bbo.sort_values("timestamp").reset_index(drop=True).copy()
    bb["mid"] = 0.5 * (bb["bid_price"] + bb["ask_price"])
    tt["s"]   = tt["side"].map({"buy": 1, "sell": -1}).astype(np.int8)
    tt["notional"] = tt["price"] * tt["amount"]
    tt["w"]   = np.minimum(tt["notional"], notional_cap)

    tt = pd.merge_asof(tt, bb[["timestamp","mid"]].rename(columns={"mid":"mid_now"}),
                       on="timestamp", direction="backward")

    bb_idx = bb["timestamp"].values
    bb_mid = bb["mid"].values
    for h_s in horizons_s:
        h_us  = int(h_s * US_PER_S)
        target = tt["timestamp"].values + h_us
        idx = np.searchsorted(bb_idx, target, side="right") - 1
        ok  = (idx >= 0) & (target <= bb_idx[-1])
        m_fut = np.where(ok, bb_mid[np.clip(idx, 0, len(bb_idx)-1)], np.nan)
        tt[f"m_fut_{h_s}"] = m_fut
        pnl = -tt["s"] * (m_fut - tt["price"]) / tt["price"] * 1e4 + 0.5
        tt[f"pnl_{h_s}_bps"] = pnl
    return tt

mk = compute_markouts(trades, bbo)
print(mk[["timestamp","side","price","notional","mid_now",
          "m_fut_30","pnl_30_bps","m_fut_120","pnl_120_bps","m_fut_300","pnl_300_bps"]].head())


In [ ]:
def weighted_mean(pnl, w):
    m = pnl.notna()
    return np.average(pnl[m], weights=w[m]) if m.any() else np.nan

print("Baseline PnL_all(tau) for", EV_DAY, "—", SYM.upper())
for h in (30, 120, 300):
    pnl_col = f"pnl_{h}_bps"
    val = weighted_mean(mk[pnl_col], mk["w"])
    n   = mk[pnl_col].notna().sum()
    print(f"  tau={h:3d}s   PnL_all = {val:+.3f} bps   (kept_trades={n:,}, "
          f"turnover=${mk.loc[mk[pnl_col].notna(),'w'].sum()/1e6:,.1f} M)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h in zip(axes, (30, 120, 300)):
    s = mk[f"pnl_{h}_bps"].dropna()
    s = s.clip(s.quantile(0.001), s.quantile(0.999))
    ax.hist(s, bins=100)
    ax.axvline(0, color="k", lw=0.5)
    ax.axvline(weighted_mean(mk[f"pnl_{h}_bps"], mk["w"]), color="r", lw=1, label="weighted mean")
    ax.set_title(f"pnl distribution, tau={h}s, {SYM.upper()}")
    ax.set_xlabel("bps"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Demo: наивный фильтр "выключить maker в окне +-2s после ликвидации того же знака".
def naive_liq_filter(trades_mk, liq_b, liq_y_shifted, window_s=2.0):
    out = pd.Series(0, index=trades_mk.index)
    ts = trades_mk["timestamp"].values
    for liq_df in (liq_b, liq_y_shifted):
        for sgn in ("buy", "sell"):
            ev = liq_df.loc[liq_df["side"] == sgn, "timestamp"].values
            if len(ev) == 0: continue
            w  = int(window_s * US_PER_S)
            mask_side = trades_mk["side"].values == sgn
            i = np.searchsorted(ev, ts, side="right") - 1
            close_left  = (i >= 0) & ((ts - ev[np.clip(i, 0, len(ev)-1)]) <= w)
            j = np.searchsorted(ev, ts, side="left")
            close_right = (j < len(ev)) & ((ev[np.clip(j, 0, len(ev)-1)] - ts) <= w)
            close = close_left | close_right
            out.loc[mask_side & close] = 1
    return out.values

f = naive_liq_filter(mk, liq_b, liq_y_shifted, window_s=2.0)
print(f"Filter window +/-2s after same-sign liq:  filtered {f.sum():,} of {len(f):,}  ({f.mean():.1%})")

rows = []
for h in (30, 120, 300):
    pnl_col = f"pnl_{h}_bps"
    ok = mk[pnl_col].notna().values
    pnl = mk.loc[ok, pnl_col].values
    w   = mk.loc[ok, "w"].values
    keep = (f[ok] == 0)
    filt = ~keep
    pnl_all  = np.average(pnl, weights=w)
    pnl_kept = np.average(pnl[keep], weights=w[keep]) if keep.any() else np.nan
    pnl_filt = np.average(pnl[filt], weights=w[filt]) if filt.any() else np.nan
    rows.append([h, pnl_all, pnl_kept, pnl_filt, pnl_kept - pnl_all,
                 w[keep].sum() / 1e6])
print("\nNaive +/-2s same-sign liq filter on", EV_DAY)
print(pd.DataFrame(rows,
    columns=["tau_s","PnL_all_bps","PnL_kept_bps","PnL_filtered_bps","Score_bps","kept_M$"])
    .round(3).to_string(index=False))


---

### Что дальше

1. **Feature pipeline** — мульти-окные TFI/IMB/PRET из *Fragmentation… in Bitcoin Markets*
   и пропагаторные суммы по ликвидациям из *Lillo, Order flow and price formation* (TIM/HDIM).
2. **Train/val split** — `2025-12-01 → 2026-01-31` / `2026-02-01 → 2026-02-28`.
3. **Модель** — взвешенная (sample_weight = w_i) регрессия `pnl_i(τ)`.
4. **Калибровка порога** на validation — максимизируем `Score(τ)` при `KeptTurnoverPerDay >= 500k`.
5. **Submission** — обёртка `(trades, bbo, liq_binance, liq_bybit) -> arrays 0/1`.
